In [2]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')


In [48]:

df = pd.read_csv("./data/kickstarter_projects.csv")
df.columns = df.columns.str.lower()
print(df.columns)
print(df.shape)
#print(df.head())
print(df.info())
df.isna().sum()
df.tail()

Index(['id', 'name', 'category', 'subcategory', 'country', 'launched',
       'deadline', 'goal', 'pledged', 'backers', 'state'],
      dtype='object')
(374853, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 374853 entries, 0 to 374852
Data columns (total 11 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   id           374853 non-null  int64 
 1   name         374853 non-null  object
 2   category     374853 non-null  object
 3   subcategory  374853 non-null  object
 4   country      374853 non-null  object
 5   launched     374853 non-null  object
 6   deadline     374853 non-null  object
 7   goal         374853 non-null  int64 
 8   pledged      374853 non-null  int64 
 9   backers      374853 non-null  int64 
 10  state        374853 non-null  object
dtypes: int64(4), object(7)
memory usage: 31.5+ MB
None


,id,name,category,subcategory,country,launched,deadline,goal,pledged,backers,state
374848,1486845240,Americas Got Talent - Serious MAK,Music,Hip-Hop,United States,2018-01-02 14:13:09,2018-01-16,500,0,0,Live
374849,974738310,EVO Planner: The World's First Personalized Fl...,Design,Product Design,United States,2018-01-02 14:15:38,2018-02-09,15000,269,8,Live
374850,2106246194,"Help save La Gattara, Arizona's first Cat Cafe!",Food,Food,United States,2018-01-02 14:17:46,2018-01-16,10000,165,3,Live
374851,1830173355,Digital Dagger Coin,Art,Art,United States,2018-01-02 14:38:17,2018-02-01,650,7,1,Live
374852,1339173863,Spirits of the Forest,Games,Tabletop Games,Spain,2018-01-02 15:02:31,2018-01-26,24274,4483,82,Live


# Data cleaningin and feature engineering

In [49]:
y = df['state'].map({'Successful': 1, 'Failed': 0, 'Live':3, 'Canceled':0, 'Suspended':5})
try:    
    X = df.drop(columns=['state'], axis = 1)
except:
    pass   

print(X.shape)
X.id.unique().size
if  y.isna().sum() != 0:
    print("There are missing values in the target variable 'state'. Removing corresponding rows from the dataset.")
    missing_indices = y[y.isna()].index
    X = X.drop(index=missing_indices).reset_index(drop=True)
    y = y.drop(index=missing_indices).reset_index(drop=True)
    print(f"New shape of X: {X.shape}, New shape of y: {y.shape}")



(374853, 10)


In [50]:
#Replace string values in a column by numbers
def StrColToNumber(dataframe, column_name):
    try:
        entries = dataframe[column_name].unique().tolist()
    except:
        return
        
    mapping = dict()
    for entry in entries:
        mapping[entry] = entries.index(entry)
    dataframe[column_name] = dataframe[column_name].map(mapping)
    

In [51]:
StrColToNumber(X, 'country')


In [52]:
StrColToNumber(X, 'category')


In [53]:
StrColToNumber(X, 'subcategory')
X.head()

,id,name,category,subcategory,country,launched,deadline,goal,pledged,backers
0,1860890148,Grace Jones Does Not Give A F$#% T-Shirt (limi...,0,0,0,2009-04-21 21:02:48,2009-05-31,1000,625,30
1,709707365,CRYSTAL ANTLERS UNTITLED MOVIE,1,1,0,2009-04-23 00:07:53,2009-07-20,80000,22,3
2,1703704063,drawing for dollars,2,2,0,2009-04-24 21:52:03,2009-05-03,20,35,3
3,727286,Offline Wikipedia iPhone app,3,3,0,2009-04-25 17:36:21,2009-07-14,99,145,25
4,1622952265,Pantshirts,0,0,0,2009-04-27 14:10:39,2009-05-26,1900,387,10


In [46]:
# split the data into training and testing sets

try:
    X.drop(columns = ['id'], axis = 1, inplace = True)
    X.reindex(drop = True, inplace = True)
except: 
    pass




In [54]:
print(X.isna().sum())
print( y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0xdeadbeef, stratify = y)

X_train.head()

id             0
name           0
category       0
subcategory    0
country        0
launched       0
deadline       0
goal           0
pledged        0
backers        0
dtype: int64
0         0
1         0
2         1
3         1
4         0
         ..
374848    3
374849    3
374850    3
374851    3
374852    3
Name: state, Length: 374853, dtype: int64


,id,name,category,subcategory,country,launched,deadline,goal,pledged,backers
157080,1680678063,Mickey Utley Band CD Project,7,36,0,2014-07-20 23:53:14,2014-08-19,24000,1205,4
154923,452115012,BLACK SHEEP TUTU fashion crafted for kids who ...,0,0,0,2014-07-14 22:23:15,2014-08-22,2500,25,1
314135,829514909,Yonderbird's Debut EP - “Destination Yet Unknown”,7,27,0,2016-10-28 23:35:03,2016-11-27,2300,3369,72
122718,871902037,"Bella, Mortar and Pestle",10,22,3,2013-12-13 06:28:23,2014-02-11,31611,23866,175
18882,2070297110,Molten Metal Extravaganza!,2,14,0,2011-04-22 06:32:18,2011-05-22,500,625,15


In [9]:
X_train.describe().T
#find the outliers in the numerical columns
num_columns = X_train.select_dtypes(include=['int64', 'float64']).columns
num_columns.drop
num_columns


Index(['id', 'goal', 'pledged', 'backers'], dtype='object')

In [ ]:

countplots = []
for i, col in enumerate(X_train.select_dtypes(include=['object']).columns):
    #pass
    plt.figure(i)
    countplots.append(X_train[col])
    plt.title(f'Distribution of {col}')

for cp in countplots:
    sns.countplot(y=cp)
plt.show()



KeyboardInterrupt: 

In [6]:
#for this exercise we will only deal with numeric variables

X = coffee_features.select_dtypes(['number'])

## Splitting data for testing 

In [8]:
#dropping Quakers column and unnamed
#changing one of the altitude to log and droping the original
X_train["altitude_mean_log"] = np.log(X_train["altitude_mean_meters"])
X_train.drop(['altitude_mean_meters'], axis=1, inplace=True)
X_train.drop(['Quakers'], axis=1, inplace=True)
X_train.drop(['Unnamed: 0'], axis=1, inplace=True)

In [9]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 22 entries, 17 to 6
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Number.of.Bags        22 non-null     int64  
 1   Harvest.Year          22 non-null     int64  
 2   Fragrance...Aroma     22 non-null     float64
 3   Flavor                22 non-null     float64
 4   Aftertaste            22 non-null     float64
 5   Salt...Acid           22 non-null     float64
 6   Bitter...Sweet        22 non-null     float64
 7   Mouthfeel             22 non-null     float64
 8   Uniform.Cup           22 non-null     float64
 9   Clean.Cup             22 non-null     float64
 10  Balance               22 non-null     float64
 11  Cupper.Points         22 non-null     float64
 12  Total.Cup.Points      22 non-null     float64
 13  Moisture              22 non-null     float64
 14  Category.One.Defects  22 non-null     int64  
 15  Category.Two.Defects  22 

In [10]:
altitude_low_meters_mean = X_train["altitude_low_meters"].mean()
altitude_high_meters_mean = X_train["altitude_high_meters"].mean()
altitude_mean_log_mean = X_train["altitude_mean_log"].mean()

In [11]:
# fillna with mean.. 
X_train["altitude_low_meters"] = X_train["altitude_low_meters"].fillna(altitude_low_meters_mean)
X_train["altitude_high_meters"] = X_train["altitude_high_meters"].fillna(altitude_high_meters_mean)
X_train["altitude_mean_log"] = X_train["altitude_mean_log"].fillna(altitude_mean_log_mean)

In [12]:
print(f"altitude low meters mean is {altitude_low_meters_mean}")
print(f"altitude_high_meters_mean is {altitude_high_meters_mean}")
print(f"altitude_mean_log_mean is {altitude_mean_log_mean}")

altitude low meters mean is 1500.3684210526317
altitude_high_meters_mean is 1505.6315789473683
altitude_mean_log_mean is 7.0571530664031155


## Trainining the model

In [13]:
## in order to exemplify how the predict will work.. we will save the y_train
X_test.to_csv("data/X_test.csv")
y_test.to_csv("data/y_test.csv")

In [15]:
#training the model
from sklearn.linear_model import LinearRegression
reg = LinearRegression().fit(X_train, y_train)

In [17]:
from sklearn.metrics import mean_squared_error
y_train_pred = reg.predict(X_train)
mse = mean_squared_error(y_train, y_train_pred)
print(mse)

6.701014816713759e-28


In [18]:
#dropping Quakers column and unnamed
#changing one of the altitude to log and droping the original
X_test["altitude_mean_log"] = np.log(X_test["altitude_mean_meters"])
X_test.drop(['altitude_mean_meters'], axis=1, inplace=True)
X_test.drop(['Quakers'], axis=1, inplace=True)
X_test.drop(['Unnamed: 0'], axis=1, inplace=True)
# fillna with mean.. 
X_test["altitude_low_meters"] = X_test["altitude_low_meters"].fillna(altitude_low_meters_mean)
X_test["altitude_high_meters"] = X_test["altitude_high_meters"].fillna(altitude_high_meters_mean)
X_test["altitude_mean_log"] = X_test["altitude_mean_log"].fillna(altitude_mean_log_mean)

In [19]:
y_test_pred = reg.predict(X_test)
mse = mean_squared_error(y_test, y_test_pred)
print(mse)

2.08680004794465e-27
